In [1]:
!nvidia-smi

Tue Dec 23 20:46:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!apt-get -qq update
!apt-get -qq install -y build-essential cmake ninja-build

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [6]:
!pip -q install -U scikit-build-core cmake ninja pybind11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 60.8 MB/s eta 0:00:00:00:0100:01


In [8]:
!which nvcc || echo "nvcc NOT found"
!ls -la /usr/local/cuda/bin/nvcc 2>/dev/null || true

/usr/local/cuda/bin/nvcc
-rwxr-xr-x 1 root root 23042472 Jun  6  2024 /usr/local/cuda/bin/nvcc


In [10]:
!ldconfig -p | grep -E "libcuda\.so" || true
!find /usr -name "libcuda.so*" 2>/dev/null | head

/usr/local/nvidia/lib64/libcuda.so.570.172.08
/usr/local/nvidia/lib64/libcuda.so
/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/cuda-12.5/compat/libcuda.so.1
/usr/local/cuda-12.5/compat/libcuda.so
/usr/local/cuda-12.5/compat/libcuda.so.555.42.06


In [13]:
!CMAKE_ARGS="-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=60 -G Ninja \
-DCUDAToolkit_ROOT=/usr/local/cuda \
-DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64" \
FORCE_CMAKE=1 \
pip install -U --no-binary=:all: llama-cpp-python --no-build-isolation

  Using cached llama_cpp_python-0.3.16.tar.gz (50.7 MB)
  Preparing metadata (pyproject.toml) ... done
  Using cached diskcache-5.6.3-py3-none-any.whl
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=52714240 sha256=2dba8225583df2047fa26d8af89b9bdc3635a6e41b8442e971ed3887b043c8db
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [ ]:
# CPU version
# !pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

In [14]:
import llama_cpp
from llama_cpp import llama_cpp as lc

print("llama-cpp-python version:", getattr(llama_cpp, "__version__", "unknown"))
print("supports_gpu_offload:", lc.llama_supports_gpu_offload())

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla P100-PCIE-16GB, compute capability 6.0, VMM: yes


llama-cpp-python version: 0.3.16
supports_gpu_offload: True


In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [54]:
from huggingface_hub import hf_hub_download

repo_id = "lapa-llm/lapa-v0.1.2-instruct-GGUF"
filename = "lapa-v0.1.2-instruct-Q4_K_M.gguf"  # try Q4_K_M first

model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(model_path)

/root/.cache/huggingface/hub/models--lapa-llm--lapa-v0.1.2-instruct-GGUF/snapshots/19e062070c68c4a06020d09b65481567b7152947/lapa-v0.1.2-instruct-Q4_K_M.gguf


In [81]:
del llm

In [82]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_batch=512,
    n_threads=4,
    verbose=False,
)

def predict(prompt, max_tokens=256):
    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=max_tokens,
    )
    return resp["choices"][0]["message"]["content"]



llama_context: n_ctx_per_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


In [97]:
import time

t_start = time.time()

prompt = """
Як рекомендовано приймати ретаболіл дорослим?
A внутрішньо
B підшкірно
C орально
D внутрішньовенно
E внутрішньом’язово
F інгаляційно
"""

print(predict(prompt))

t_end = time.time()
print("time taken: ", t_end - t_start)

E внутрішньом’язово
time taken:  0.2425980567932129


In [85]:
!nvidia-smi

Tue Dec 23 21:22:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             43W /  250W |    9397MiB /  16384MiB |     68%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----